In [28]:
# 1. ALL IMPORTS
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Optional for warnings
import warnings
warnings.filterwarnings('ignore')

In [29]:
# 2. LOAD COMBINED DATASET
folder = r"C:\Users\omars\Downloads"
files = [
    "NYC_Taxi_Dataset_October.csv",
    "NYC_Taxi_Dataset_November.csv",
    "NYC_Taxi_Dataset_December.csv"
]

dtype_settings = {
    "passenger_count": "str",
    "RatecodeID": "str",
    "congestion_surcharge": "str",
    "Airport_fee": "str"
}

dfs = []
for file in files:
    path = os.path.join(folder, file)
    print("Loading:", path)
    df = pd.read_csv(path, dtype=dtype_settings, low_memory=False)
    df["source_month"] = file.split("_")[-1].replace(".csv", "")
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
print("Initial shape:", combined_df.shape)

Loading: C:\Users\omars\Downloads\NYC_Taxi_Dataset_October.csv
Loading: C:\Users\omars\Downloads\NYC_Taxi_Dataset_November.csv
Loading: C:\Users\omars\Downloads\NYC_Taxi_Dataset_December.csv
Initial shape: (10238567, 20)


In [30]:
# 3. CLEAN MISSING VALUES & \N
combined_df = combined_df.replace(r'\\N', np.nan, regex=True)

num_cols = [
    "passenger_count", "trip_distance", "fare_amount", "extra", "mta_tax",
    "tip_amount", "tolls_amount", "improvement_surcharge",
    "total_amount", "congestion_surcharge", "Airport_fee"
]

for col in num_cols:
    combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')

# Drop rows with any NaN in numeric columns
combined_df = combined_df.dropna(subset=num_cols)
print("After cleaning missing values:", combined_df.shape)

After cleaning missing values: (9770960, 20)


In [31]:
# 4. CONVERT DATETIME TO NUMERIC FEATURES
combined_df["tpep_pickup_datetime"] = pd.to_datetime(combined_df["tpep_pickup_datetime"])
combined_df["tpep_dropoff_datetime"] = pd.to_datetime(combined_df["tpep_dropoff_datetime"])

# Extract useful numeric features
combined_df["pickup_hour"] = combined_df["tpep_pickup_datetime"].dt.hour
combined_df["pickup_day"] = combined_df["tpep_pickup_datetime"].dt.day
combined_df["trip_duration"] = (combined_df["tpep_dropoff_datetime"] - combined_df["tpep_pickup_datetime"]).dt.total_seconds() / 60

# Drop original datetime columns
combined_df = combined_df.drop(columns=["tpep_pickup_datetime", "tpep_dropoff_datetime"])

In [32]:
# 5. REMOVE OUTLIERS (SAFE)
def remove_outliers(df, column, factor=3):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

for col in ["trip_distance", "fare_amount", "total_amount", "tip_amount", "trip_duration"]:
    combined_df = remove_outliers(combined_df, col, factor=3)

print("After outlier removal:", combined_df.shape)

After outlier removal: (8674609, 21)


In [33]:
# 6. ENCODE CATEGORICAL VARIABLES
cat_cols = ["VendorID", "RatecodeID", "store_and_fwd_flag",
            "payment_type", "PULocationID", "DOLocationID"]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined_df[col] = le.fit_transform(combined_df[col].astype(str))
    encoders[col] = le

In [34]:
# 7. SAVE CLEANED DATASET
output_path = os.path.join(folder, "NYC_Taxi_Cleaned_Final_Version.csv")
combined_df.to_csv(output_path, index=False)
print("Cleaned dataset saved to:", output_path)

Cleaned dataset saved to: C:\Users\omars\Downloads\NYC_Taxi_Cleaned_Final_Version.csv


In [1]:
#Basic shape
print("Dataset shape (rows, columns):", df.shape)

NameError: name 'df' is not defined